# 01 — Ingest Dokumen

**Tujuan:** mengumpulkan bahan penelitian: memindai PDF LPMB yang sudah tersedia di
`data/raw/pdf/`, mengekstraksi teksnya, dan menghasilkan katalog dokumen (§6 `perancangan.md`).

Notebook ini TIDAK mengunduh dari internet — sumber PDF sudah ditaruh manual di
`data/raw/pdf/` sebelum notebook dijalankan. Fungsi unduh (`ingest.download_pdf`,
`ingest.snapshot_web_page`) tetap tersedia di `src/ingest.py` untuk dipakai nanti bila
korpus diperluas lewat `configs/sources.yaml`.

**INPUT:** `../data/raw/pdf/*.pdf`

**OUTPUT:** `../data/interim/{doc_id}.txt` (teks per halaman), `../data/processed/documents.jsonl`

**Perkiraan runtime:** beberapa detik per 100 halaman PDF

**Perkiraan biaya API:** Rp 0 — tidak ada panggilan LLM di notebook ini

**Assert di notebook ini:** setiap PDF di `data/raw/pdf/` menghasilkan tepat satu baris
di `documents.jsonl`; `doc_id` unik; `sha256` terisi; `local_path` menunjuk berkas yang ada.

In [13]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "../src")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import pandas as pd

from config import load_config, set_seed

cfg = load_config("../configs/base.yaml")
set_seed(cfg.seed)

TypeError: ChunkingConfig.__init__() got an unexpected keyword argument 'max_dot_ratio'

## Pindai PDF lokal & hitung sha256

`sha256` dipakai untuk mendeteksi duplikat (dua nama berkas berbeda, isi identik).

In [3]:
import ingest

pdf_paths = ingest.scan_local_pdfs("../data/raw/pdf")
hashes = {p: ingest.sha256_of_file(p) for p in pdf_paths}

dupes = pd.Series(list(hashes.values())).value_counts()
dupes = dupes[dupes > 1]
if len(dupes):
    print("PERINGATAN — sha256 duplikat terdeteksi (kemungkinan berkas sama diunggah dua kali):")
    for h in dupes.index:
        print(" ", h, [p.name for p in hashes if hashes[p] == h])

print(f"{len(pdf_paths)} PDF ditemukan di data/raw/pdf/")

1 PDF ditemukan di data/raw/pdf/


## Metadata dokumen (EDIT bila perlu)

`program`/`year`/`title` ditebak otomatis dari nama berkas. Bila nama berkas kriptik
(mis. `"794KR PPENGMAS PKM DPR 2026"`), override di `metadata_overrides` sebelum
melanjutkan — kunci dict adalah nama berkas persis seperti di `data/raw/pdf/`.

In [4]:
import re


def guess_metadata(filename: str) -> dict:
    stem = re.sub(r"\s*\(\d+\)$", "", Path(filename).stem)  # buang sufiks " (2)" dsb
    year_match = re.search(r"20\d{2}", stem)
    year = int(year_match.group()) if year_match else pd.Timestamp.now().year
    return {"title": stem, "program": "UNKNOWN", "year": year}


from pathlib import Path

# EDIT di sini: kunci = nama berkas persis di data/raw/pdf/, nilai = override
# metadata (title/program/year). Baris yang tidak disebutkan memakai tebakan
# otomatis dari guess_metadata() di atas.
metadata_overrides = {
    "Panduan Usulan Pengmas Internal Universitas Airlangga Tahun 2026 (2).pdf": {
        "title": "Panduan Usulan Pengmas Internal Universitas Airlangga Tahun 2026",
        "program": "Pengmas Internal",
        "year": 2026,
    },
}

In [5]:
rows = []
for p in pdf_paths:
    meta = {**guess_metadata(p.name), **metadata_overrides.get(p.name, {})}
    rows.append({"filename": p.name, "local_path": p, "sha256": hashes[p], **meta})

meta_df = pd.DataFrame(rows)
meta_df

,filename,local_path,sha256,title,program,year
0,Panduan Usulan Pengmas Internal Universitas Ai...,..\data\raw\pdf\Panduan Usulan Pengmas Interna...,16462b0984d92320b8d08cebe2912e11fd57286680dba2...,Panduan Usulan Pengmas Internal Universitas Ai...,Pengmas Internal,2026


## Ekstraksi teks (PyMuPDF + pdfplumber untuk halaman bertabel)

In [6]:
extracted = {}  # local_path -> list[{"page", "text", "method"}]
for row in meta_df.itertuples():
    pages = ingest.extract_text_auto(row.local_path)
    extracted[row.local_path] = pages
    n_table_pages = sum(1 for pg in pages if pg["method"] == "pdfplumber_table")
    print(f"{row.filename}: {len(pages)} halaman, {n_table_pages} lewat pdfplumber (tabel)")

Panduan Usulan Pengmas Internal Universitas Airlangga Tahun 2026 (2).pdf: 99 halaman, 47 lewat pdfplumber (tabel)


## Rakit record dokumen & tulis teks interim

Dokumen dengan rata-rata < 20 karakter/halaman ditandai `extractable=false` (kemungkinan
besar gambar/poster) TETAPI tetap dicatat di `documents.jsonl`, bukan dibuang
(Batasan Masalah #2).

In [7]:
existing_ids: set[str] = set()
records = []
not_extractable = []

for row in meta_df.itertuples():
    pages = extracted[row.local_path]
    doc_id = ingest.make_doc_id(row.title, row.year, existing_ids)
    existing_ids.add(doc_id)

    extractable = ingest.is_extractable(pages)
    if not extractable:
        not_extractable.append(row.filename)

    ingest.write_interim_text(doc_id, pages, "../data/interim")

    records.append(
        ingest.build_document_record(
            doc_id=doc_id,
            title=row.title,
            source_type="pdf",
            source_url=None,
            local_path=Path(*Path(row.local_path).parts[1:]).as_posix(),  # root-relative, bukan relatif ke notebooks/
            program=row.program,
            year=row.year,
            n_pages=len(pages),
            sha256=row.sha256,
            extractable=extractable,
        )
    )

print(f"{len(records)} dokumen diproses, {len(not_extractable)} ditandai extractable=false")
if not_extractable:
    print("Tidak extractable:", not_extractable)

1 dokumen diproses, 0 ditandai extractable=false


## ASSERT

In [8]:
assert len(records) == len(pdf_paths), "Jumlah record tidak sama dengan jumlah PDF sumber"

doc_ids = [r["doc_id"] for r in records]
assert len(doc_ids) == len(set(doc_ids)), "doc_id tidak unik"

for r in records:
    assert r["sha256"], f"sha256 kosong untuk {r['doc_id']}"
    assert Path("..", r["local_path"]).exists(), f"local_path tidak ada: {r['local_path']}"
    assert r["n_pages"] > 0, f"n_pages nol untuk {r['doc_id']}"

print("Semua assert lolos.")

Semua assert lolos.


## Tulis `documents.jsonl`

In [9]:
out_path = "../data/processed/documents.jsonl"
ingest.write_documents_jsonl(records, out_path)
print(f"Ditulis {len(records)} record ke {out_path}")

Ditulis 1 record ke ../data/processed/documents.jsonl


## Ringkasan

In [10]:
summary_df = pd.DataFrame(records)[
    ["doc_id", "title", "program", "year", "n_pages", "extractable"]
]
print(f"Total dokumen: {len(summary_df)}")
print(f"Total halaman: {summary_df['n_pages'].sum()}")
print(f"Tidak extractable: {(~summary_df['extractable']).sum()}")
summary_df

Total dokumen: 1
Total halaman: 99
Tidak extractable: 0


,doc_id,title,program,year,n_pages,extractable
0,lpmb-2026-panduan-usulan-pengmas-internal-univ...,Panduan Usulan Pengmas Internal Universitas Ai...,Pengmas Internal,2026,99,True
